In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Reload data
print("Loading data...")
train_trans = pd.read_csv('train_transaction.csv')
train_id = pd.read_csv('train_identity.csv')
df = train_trans.merge(train_id, on='TransactionID', how='left')
df = df.copy()
print(f"Shape: {df.shape}")

# Feature engineering
print("\nEngineering features...")

# 1. Time features
df['hour'] = (df['TransactionDT'] / 3600 % 24).astype(int)
df['day'] = (df['TransactionDT'] / (3600*24) % 7).astype(int)
df['is_night'] = ((df['hour'] >= 0) & (df['hour'] <= 9)).astype(int)

# 2. Log transform
df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

# 3. High risk email flag
high_risk_emails = ['protonmail.com', 'outlook.com', 'hotmail.com']
df['is_highrisk_email'] = df['P_emaildomain'].isin(high_risk_emails).astype(int)

# 4. Has identity flag
df['has_identity'] = df['id_01'].notna().astype(int)

print("Features engineered:")
print("  - hour, day, is_night")
print("  - TransactionAmt_log")
print("  - is_highrisk_email")
print("  - has_identity")
print("\nDone.")

Loading data...
Shape: (590540, 434)

Engineering features...
Features engineered:
  - hour, day, is_night
  - TransactionAmt_log
  - is_highrisk_email
  - has_identity

Done.


In [2]:
# Feature selection - drop high missing + irrelevant columns
print("Selecting features...")

# Drop columns with >80% missing
missing_pct = df.isnull().sum() / len(df)
high_missing_cols = missing_pct[missing_pct > 0.8].index.tolist()
print(f"Dropping {len(high_missing_cols)} columns with >80% missing")

# Drop irrelevant columns
drop_cols = ['TransactionID', 'TransactionDT'] + high_missing_cols
df_model = df.drop(columns=drop_cols)
print(f"Shape after dropping: {df_model.shape}")

# Encode categorical columns
print("\nEncoding categorical features...")
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {len(cat_cols)}")

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = df_model[col].astype(str)
    df_model[col] = le.fit_transform(df_model[col])

# Fill remaining missing values
df_model = df_model.fillna(-999)
print("Missing values filled with -999")

# Final shape
print(f"\nFinal model dataframe shape: {df_model.shape}")
print(f"Target (isFraud) distribution:\n{df_model['isFraud'].value_counts()}")

Selecting features...
Dropping 74 columns with >80% missing
Shape after dropping: (590540, 364)

Encoding categorical features...
Categorical columns: 26
Missing values filled with -999

Final model dataframe shape: (590540, 364)
Target (isFraud) distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64


In [ ]:
# Train/Test split
print("Splitting data...")

X = df_model.drop(columns=['isFraud'])
y = df_model['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nTrain fraud rate: {y_train.mean()*100:.2f}%")
print(f"Test fraud rate:  {y_test.mean()*100:.2f}%")

# Save processed data
print("\nSaving processed data...")
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("\nFeature Engineering complete. Ready for Model Training.")

Splitting data...
X_train: (472432, 363)
X_test:  (118108, 363)

Train fraud rate: 3.50%
Test fraud rate:  3.50%

Saving processed data...
